In [4]:
from dotenv import load_dotenv
from agents import Agent, Runner, ModelSettings, handoff

In [ ]:
    input_length_guardrail,
    output_faithfulness_guardrail,
    output_length_guardrail,
    pii_output_guardrail,

In [ ]:
tools

    generate_answer,
    retrieve_context,
    understand_query,

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

from src.query.understanding import ProcessedQuery
from src.generation.generator import GenerationResult


@dataclass
class RAGRunContext:
    """Shared mutable context threaded through the entire agent pipeline."""

    # ── Input ──────────────────────────────────────────────────────────
    raw_query: str
    conversation_history: List[Dict[str, str]] = field(default_factory=list)
    auth_token: Optional[str] = None
    correlation_id: str = ""
    namespace: str = "default"

    # ── Populated progressively by agents ──────────────────────────────
    routing_decision = None
    corrected_query: Optional[str] = None   # written by spell_correct_and_clarify tool
    processed_query: Optional[ProcessedQuery] = None
    context_items: List[Any] = field(default_factory=list)   # List[ContextItem]
    generation_result: Optional[GenerationResult] = None

    # ── Diagnostics ────────────────────────────────────────────────────
    agent_trace: List[str] = field(default_factory=list)

    def record(self, agent_name: str, event: str) -> None:
        """Append a trace entry (agent_name: event) for observability."""
        self.agent_trace.append(f"[{agent_name}] {event}")

In [ ]:
ConversationalAgent: Agent[RAGRunContext] = Agent(
    name="ConversationalAgent",
    handoff_description=(
        "Handles greetings, casual conversation, and general non-retrieval interactions."
    ),
    instructions=(
        "You are a friendly and concise conversational assistant.\n\n"
        "The user's message does not require document retrieval or external knowledge lookup.\n"
        "Respond naturally, conversationally, and briefly.\n\n"
        "Rules:\n"
        "- Keep responses short and warm.\n"
        "- Do not fabricate factual information.\n"
        "- If the user appears to ask a knowledge-intensive question, "
        "encourage them to provide more details instead of answering from prior knowledge.\n"
        "- Never claim to have retrieved information."
    ),
    model=_orchestrator_model,
    model_settings=ModelSettings(temperature=0.7),
    output_type=DirectResponseOutput,
)



RetrievalAgent: Agent[RAGRunContext] = Agent(
    name="RetrievalAgent",
    handoff_description=(
        "Handles queries requiring retrieval from external knowledge sources."
    ),
    instructions=(
        "You are a retrieval-augmented generation (RAG) assistant.\n\n"
        "Your task is to answer the user's question using ONLY retrieved context.\n\n"
        "Execution flow (mandatory):\n\n"
        "STEP 1 — Call understand_query.\n"
        "- Rewrite the query into a fully self-contained form.\n"
        "- Generate sub-questions if necessary.\n"
        "- Prepare metadata filters.\n\n"
        "STEP 2 — Call retrieve_context.\n"
        "- Retrieve relevant passages for the main query and sub-questions.\n"
        "- Never answer before retrieval.\n\n"
        "STEP 3 — Call generate_answer.\n"
        "- Generate a grounded answer strictly from retrieved context.\n\n"
        "Rules:\n"
        "- NEVER use prior knowledge.\n"
        "- NEVER fabricate information.\n"
        "- Preserve all [CITE:...] markers exactly.\n"
        "- If retrieval is insufficient, return the generated fallback response exactly.\n"
        "- If conflict information exists, include it verbatim.\n"
        "- Do not add extra commentary, disclaimers, or preambles."
    ),
    tools=[understand_query, retrieve_context, generate_answer],
    model=_worker_model,
    model_settings=ModelSettings(temperature=0.0),
    output_type=GenerationOutput,
    output_guardrails=[
        output_faithfulness_guardrail,
        output_length_guardrail,
    ],
)



RefinementAgent: Agent[RAGRunContext] = Agent(
    name="RefinementAgent",
    handoff_description=(
        "Handles follow-up questions and response refinements using previously "
        "retrieved context before deciding whether new retrieval is necessary."
    ),
    instructions=(
        "You are a contextual refinement assistant.\n\n"
        "Your PRIMARY goal is to answer from conversation history WITHOUT retrieval.\n\n"
        "STEP 1 — Check conversation history first (MANDATORY).\n"
        "- Read the <conversation_history> carefully.\n"
        "- If the answer is already present in the history, answer directly.\n"
        "- Short follow-ups like 'and degree?', 'his highest degree', 'what about X?', "
        "'I see', 'tell me more' almost ALWAYS refer to the previous answer — use it.\n"
        "- Only proceed to retrieval if the topic has clearly shifted to something "
        "NOT mentioned anywhere in the conversation history.\n\n"
        "STEP 2 — Decide: can history answer this?\n"
        "- YES → call generate_answer directly using ctx.context.context_items "
        "if available, or answer inline.\n"
        "- NO (genuinely new topic) → proceed to STEP 3.\n\n"
        "STEP 3 — Only if retrieval is truly required:\n"
        "- Call understand_query.\n"
        "- Call retrieve_context.\n"
        "- Call generate_answer.\n\n"
        "Rules:\n"
        "- NEVER retrieve if the history already contains the answer.\n"
        "- NEVER fabricate information.\n"
        "- NEVER use unsupported prior knowledge.\n"
        "- Preserve all [CITE:...] markers exactly.\n"
        "- If retrieval is insufficient, return the fallback response exactly.\n"
        "- Keep answers concise and grounded in context."
    ),
    tools=[understand_query, retrieve_context, generate_answer],
    model=_worker_model,
    model_settings=ModelSettings(temperature=0.0),
    output_type=GenerationOutput,
    output_guardrails=[
        output_faithfulness_guardrail,
        output_length_guardrail,
    ],
)



OrchestratorAgent: Agent[RAGRunContext] = Agent(
    name="OrchestratorAgent",
    instructions=(
        "You are the routing controller for a multi-agent RAG system.\n\n"
        "Your ONLY responsibility is to select the correct agent.\n"
        "You must NEVER answer the user's question directly.\n\n"
        "Routing Rules:\n\n"
        "1. ConversationalAgent\n"
        "- Greetings (hi, hello, hey)\n"
        "- Pure small talk with no information need (how are you, I see, great)\n"
        "- Reactions and acknowledgements (ok, thanks, interesting)\n\n"
        "2. RetrievalAgent\n"
        "- ANY question about a person, place, thing, or concept\n"
        "- 'Do you know X?' — this is a knowledge query, NOT small talk\n"
        "- 'Tell me about X', 'Who is X', 'What is X'\n"
        "- 'Can you find information about X'\n"
        "- Any named entity (person name, company, location) in the query\n"
        "- Fact-based or knowledge-intensive requests\n\n"
        "3. RefinementAgent\n"
        "- Follow-up questions referencing a previous answer\n"
        "- 'And his degree?', 'Tell me more', 'What about X?'\n"
        "- Requests to simplify, expand, or clarify a previous answer\n"
        "- Short fragments that only make sense in context of prior turns\n\n"
        "CRITICAL RULES:\n"
        "- 'Do you know [name]?' is ALWAYS a RetrievalAgent query.\n"
        "- Any message containing a proper noun (name, place, company) "
        "that isn't pure small talk goes to RetrievalAgent or RefinementAgent.\n"
        "- When in doubt between ConversationalAgent and RetrievalAgent, "
        "always prefer RetrievalAgent.\n"
        "- Always hand off immediately. Never answer directly."
    ),
    handoffs=[
        handoff(ConversationalAgent),
        handoff(RetrievalAgent),
        handoff(RefinementAgent),
    ],
    model=_orchestrator_model,
    model_settings=ModelSettings(temperature=0.0),
    input_guardrails=[input_length_guardrail],
    output_guardrails=[pii_output_guardrail],
)

In [ ]:
import asyncio

result = asyncio.run(
    OrchestratorAgent.run(
        raw_query=question,
        # conversation_history=history,
    )
)